# Việc A — Đặt trọng số cao hơn cho chuyến hiếm

> **Mentor:** *"Giữ model hiện tại làm model mốc, sau đó đặt weight lớn hơn cho các chuyến hiếm,
> cụ thể là chuyến trên 300k hoặc trên 15 km. Kiểm tra xem sai số ở hai nhóm này giảm bao nhiêu và
> kết quả chung phải đánh đổi bao nhiêu."*

## Notebook này trả lời gì

Một câu hỏi duy nhất: **đánh đổi bao nhiêu?** Mỗi mức trọng số cho một cặp số — nhóm hiếm tốt lên
bao nhiêu, toàn tập xấu đi bao nhiêu — và ta chọn điểm trên đường đánh đổi đó.

## Hai quyết định thiết kế cần nói trước

**Chỉ train lại nhánh giá cơ bản.** Phân rã sai số cho thấy 98,9% phương sai nằm ở tầng này, còn
tầng hệ số nhân đã đạt MAPE 1,42%. Đặt trọng số ở tầng gần hoàn hảo là vô ích. Dự đoán hệ số nhân
lấy nguyên từ `uq_pred_test.parquet`.

**Hai cách gán chuyến hiếm, chạy cả hai:**

| Cách | Điều kiện | Ưu | Nhược |
|---|---|---|---|
| `quang_duong` | `quote_distance > 15 km` | Quan sát được cả lúc train lẫn lúc suy luận | Không trúng đúng nhóm giá cao |
| `gia` | giá cơ bản thật > 300k / 1,165 | Trúng đúng nhóm mentor nêu | Chỉ biết lúc train — hợp lệ, nhưng model học một khái niệm nó không quan sát được |

Đây là câu hỏi cần mentor chốt. Notebook chạy cả hai để có số mà bàn.

> ⚠️ **Nhóm đánh giá cố định** theo giá thật và quãng đường, giống nhau cho mọi phương án — đúng
> yêu cầu của mentor.

In [ ]:
import warnings, time, sys, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

BLUE, ORANGE, GREEN, RED, PURPLE, MUT = ("#0072B2", "#E69F00", "#009E73",
                                         "#D55E00", "#CC79A7", "#666666")
INK = "#222222"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .25,
    "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10.5,
})
EVAL = Path("../model/evaluation")
DATA = Path("../data/hcm_train_ready.parquet")
HINH = Path("../docs/hinh_anh"); HINH.mkdir(parents=True, exist_ok=True)
KQ   = Path("ket_qua"); KQ.mkdir(exist_ok=True)

sys.path.insert(0, "../model")
from _common_train import CAT, B_NUM, dat_categories, prep, tao_histgb

In [ ]:
# ══ NHÓM CỐ ĐỊNH — quy tắc chốt cho cả tuần 5 ══════════════════════════
# Mentor tuần 4: "giữ nguyên nhóm chuyến giữa các model. Không nên để mỗi model
# tự chia nhóm theo giá mà chính nó dự đoán."
# => nhóm chia theo GIÁ THẬT và QUÃNG ĐƯỜNG, hai thứ không phụ thuộc model nào.
CAT_GIA = [0, 50e3, 100e3, 150e3, 200e3, 300e3, np.inf]
TEN_GIA = ["<50k", "50–100k", "100–150k", "150–200k", "200–300k", ">300k"]
CAT_KM  = [0, 2, 5, 8, 12, 15, np.inf]
TEN_KM  = ["<2", "2–5", "5–8", "8–12", "12–15", ">15"]

def gan_nhom(d, cot_gia="y", cot_km="km"):
    d = d.copy()
    d["band"] = pd.cut(d[cot_gia], CAT_GIA, labels=TEN_GIA)
    d["kmb"]  = pd.cut(d[cot_km],  CAT_KM,  labels=TEN_KM)
    return d

def mape(p, y):
    p, y = np.asarray(p, float), np.asarray(y, float)
    return float(np.mean(np.abs(p - y) / y))

def boot_hieu(p_moc, p_moi, y, B=2000, seed=7):
    """CI 95% cho (sai số mốc − sai số mới). Dương = phương án mới TỐT HƠN."""
    y = np.asarray(y, float)
    h = np.abs(np.asarray(p_moc, float) - y)/y - np.abs(np.asarray(p_moi, float) - y)/y
    if len(h) < 2:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    mau = rng.integers(0, len(h), size=(B, len(h)))
    pp = h[mau].mean(axis=1)
    return h.mean(), float(np.percentile(pp, 2.5)), float(np.percentile(pp, 97.5))

def bang_theo_nhom(d, cot_moc, cot_moi, ten_moc="mốc", ten_moi="mới"):
    """MAPE hai phương án trên từng nhóm cố định + CI của chênh lệch."""
    hang = []
    for cot_nhom, nhan in [("kmb", "km"), ("band", "giá thật")]:
        for g, s in d.groupby(cot_nhom, observed=True):
            if len(s) < 30:
                continue
            m, lo, hi = boot_hieu(s[cot_moc], s[cot_moi], s.y)
            hang.append({"Chia theo": nhan, "Nhóm": str(g), "n": len(s),
                         ten_moc: mape(s[cot_moc], s.y), ten_moi: mape(s[cot_moi], s.y),
                         "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                         "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    m, lo, hi = boot_hieu(d[cot_moc], d[cot_moi], d.y)
    hang.append({"Chia theo": "—", "Nhóm": "TOÀN TẬP", "n": len(d),
                 ten_moc: mape(d[cot_moc], d.y), ten_moi: mape(d[cot_moi], d.y),
                 "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                 "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    return pd.DataFrame(hang)

def in_bang(df, cot_mape):
    return df.style.format({**{c: "{:.2%}" for c in cot_mape},
                            "Chênh (điểm)": "{:+.2f}", "CI thấp": "{:+.2f}",
                            "CI cao": "{:+.2f}", "n": "{:,}"}).hide(axis="index")

## 1. Cấu hình

`NHANH = True` lấy mẫu tập train để chạy nhanh khi thử. **Số cuối cùng đưa vào báo cáo phải chạy
với `NHANH = False`** — có in cảnh báo ở cuối notebook nếu quên.

In [ ]:
NHANH   = True          # True: lấy mẫu train cho nhanh · False: chạy đầy đủ
N_MAU   = 500_000       # số dòng train mỗi tháng khi NHANH
TRONG_SO = [1, 2, 3, 5, 10]
CACH_GAN = ["quang_duong", "gia"]

NGUONG_KM  = 15.0
NGUONG_GIA = 300_000 / 1.165     # quy về giá cơ bản: hệ số nhân TB 1,165

print(f"{'CHẾ ĐỘ NHANH — số chỉ để thử' if NHANH else 'CHẾ ĐỘ ĐẦY ĐỦ'}")
print(f"{len(CACH_GAN)} cách gán × {len(TRONG_SO)} mức trọng số × 3 tháng = "
      f"{len(CACH_GAN)*len(TRONG_SO)*3} lượt train")
print(f"Ước tính: ~{len(CACH_GAN)*len(TRONG_SO)*3*(0.6 if NHANH else 2.2):.0f} phút")

## 2. Nạp dữ liệu

In [ ]:
COLS = list(dict.fromkeys(CAT + B_NUM + [
    "target_shown_price", "target_shown_multiplier", "latest_observed_price",
    "latest_observed_multiplier", "evaluation_month", "split", "requested_lag_minutes"]))
COLS = [c for c in COLS if c != "latest_observed_base"]      # cột dẫn xuất

t0 = time.time()
df = pd.read_parquet(DATA, columns=COLS)
df = dat_categories(df)                                       # BẮT BUỘC trước khi chia tập
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = (df.latest_observed_price
                              / df.latest_observed_multiplier.clip(lower=0.1))
print(f"Nạp {len(df):,} dòng × {len(df.columns)} cột · {time.time()-t0:.0f}s")

THANGS = sorted(df.evaluation_month.unique())
print("Tháng:", THANGS)
for th in THANGS:
    s = df[df.evaluation_month == th]
    print(f"  {th}: train {(s.split=='train').sum():,} · test {(s.split=='test').sum():,}")

## 3. Lấy dự đoán hệ số nhân của model mốc

Ghép theo thứ tự hàng. Có kiểm tra khớp bằng `gia_that` — nếu lệch thì dừng ngay chứ không âm thầm
cho ra số sai.

In [ ]:
ut = pd.read_parquet(EVAL / "uq_pred_test.parquet").reset_index(drop=True)

te_all = pd.concat([df[(df.evaluation_month == th) & (df.split == "test")]
                    for th in THANGS]).reset_index(drop=True)
assert len(te_all) == len(ut), f"số dòng lệch: {len(te_all)} vs {len(ut)}"
assert np.allclose(te_all.target_shown_price.values, ut.gia_that.values), \
    "thứ tự hàng không khớp — dừng lại, đừng dùng số này"
print(f"Khớp {len(te_all):,} dòng test với uq_pred_test.parquet")

te_all["heso_pred"]   = ut.heso_pred.values
te_all["hybrid_moc"]  = ut.hybrid_pred.values      # model mốc, chưa đặt trọng số
te_all["base_moc"]    = ut.base_pred.values

## 4. Huấn luyện theo từng mức trọng số

`sample_weight` của HistGradientBoosting: chuyến hiếm được nhân trọng số `w`, chuyến còn lại giữ 1.
Trọng số chỉ áp lên **tập train**, không áp lên test.

In [ ]:
def mask_hiem(d, cach):
    if cach == "quang_duong":
        return (d.quote_distance > NGUONG_KM).values
    if cach == "gia":
        return (d.base_price > NGUONG_GIA).values
    raise ValueError(cach)

KQ_PRED = {}                                  # (cach, w) -> mang du doan gia cuoi tren te_all
rng = np.random.default_rng(42)

for cach in CACH_GAN:
    for w in TRONG_SO:
        if cach == CACH_GAN[0] and w == 1:
            pass                              # w=1 van chay de doi chieu voi model moc
        t0, phan = time.time(), []
        for th in THANGS:
            sub = df[df.evaluation_month == th]
            tr  = sub[sub.split == "train"]
            if NHANH and len(tr) > N_MAU:
                tr = tr.iloc[rng.choice(len(tr), N_MAU, replace=False)]
            te = te_all[te_all.evaluation_month == th]

            sw = np.where(mask_hiem(tr, cach), float(w), 1.0)
            m = tao_histgb()
            m.fit(prep(tr, B_NUM), np.log(tr.base_price), sample_weight=sw)
            phan.append(pd.Series(np.exp(m.predict(prep(te, B_NUM))), index=te.index))
        base_pred = pd.concat(phan).reindex(te_all.index).values
        KQ_PRED[(cach, w)] = base_pred * te_all.heso_pred.values
        ti_le = mask_hiem(df[df.split == "train"], cach).mean()
        print(f"[{cach:12} w={w:>2}] {time.time()-t0:5.0f}s · "
              f"chuyến hiếm chiếm {ti_le:.2%} tập train")
print("Xong.")

## 5. Đánh giá trên nhóm cố định

Chỉ đánh giá trên **độ trễ 5 phút** để so được với mọi số của tuần 4.

In [ ]:
m5 = te_all.requested_lag_minutes.values == 5
E = pd.DataFrame({
    "y":      te_all.target_shown_price.values[m5],
    "km":     te_all.quote_distance.values[m5],
    "moc":    te_all.hybrid_moc.values[m5],
})
for k, v in KQ_PRED.items():
    E[f"{k[0]}_w{k[1]}"] = v[m5]
E = gan_nhom(E)
print(f"{len(E):,} chuyến test lag 5 phút")
print(f"MAPE model mốc: {mape(E.moc, E.y):.2%}")

In [ ]:
# Bang danh doi: moi phuong an mot hang
NHOM_QUAN_TAM = [("kmb", ">15"), ("kmb", "12–15"), ("band", ">300k")]

hang = []
for cach in CACH_GAN:
    for w in TRONG_SO:
        cot = f"{cach}_w{w}"
        r = {"Cách gán": cach, "w": w, "Toàn tập": mape(E[cot], E.y)}
        for cn, g in NHOM_QUAN_TAM:
            s = E[E[cn].astype(str) == g]
            r[g] = mape(s[cot], s.y)
        r["Δ toàn tập (điểm)"] = (mape(E[cot], E.y) - mape(E.moc, E.y)) * 100
        hang.append(r)
moc_row = {"Cách gán": "— MỐC —", "w": "", "Toàn tập": mape(E.moc, E.y),
           "Δ toàn tập (điểm)": 0.0}
for cn, g in NHOM_QUAN_TAM:
    s = E[E[cn].astype(str) == g]
    moc_row[g] = mape(s.moc, s.y)

DA = pd.DataFrame([moc_row] + hang)
DA.style.format({c: "{:.2%}" for c in ["Toàn tập", ">15", "12–15", ">300k"]}
                | {"Δ toàn tập (điểm)": "{:+.3f}"}).hide(axis="index")

### Đọc bảng

Cột `Δ toàn tập` là **cái giá phải trả**. Cột `>15` và `>300k` là **cái nhận được**. Tiêu chí đã đặt
ở `VIEC_TUAN_5.md`: nhóm hiếm giảm ≥1 điểm trong khi toàn tập xấu đi ≤0,15 điểm.

In [ ]:
# ═════════ HÌNH TS1 — đường đánh đổi ═════════
fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))
MAU = {"quang_duong": BLUE, "gia": ORANGE}
NHAN_NHOM = [("kmb", ">15", "Chuyến > 15 km"), ("band", ">300k", "Giá thật > 300k")]

for a, (cn, g, tieu_de) in zip(ax, NHAN_NHOM):
    s_moc = E[E[cn].astype(str) == g]
    x0, y0 = mape(E.moc, E.y)*100, mape(s_moc.moc, s_moc.y)*100
    a.scatter([x0], [y0], s=150, marker="*", color=INK, zorder=5, label="model mốc")
    for cach in CACH_GAN:
        xs, ys = [], []
        for w in TRONG_SO:
            cot = f"{cach}_w{w}"
            s = E[E[cn].astype(str) == g]
            xs.append(mape(E[cot], E.y)*100); ys.append(mape(s[cot], s.y)*100)
        a.plot(xs, ys, "o-", color=MAU[cach], lw=2, ms=7, label=f"gán theo {cach}")
        for w, x, y in zip(TRONG_SO, xs, ys):
            a.annotate(f"w={w}", (x, y), fontsize=9, xytext=(4, 4),
                       textcoords="offset points", color=MAU[cach])
    a.axvline(x0, color=MUT, ls=":", lw=1)
    a.axhline(y0, color=MUT, ls=":", lw=1)
    a.set_xlabel("MAPE toàn tập (%)  →  càng phải càng tệ")
    a.set_ylabel(f"MAPE nhóm {g} (%)")
    a.set_title(tieu_de, fontweight="bold")
    a.legend(frameon=False, fontsize=9.5)

fig.suptitle("TS1 — Đường đánh đổi: được gì ở nhóm hiếm, mất gì ở toàn tập\n"
             "Góc dưới-trái là tốt. Điểm sao là model mốc chưa đặt trọng số",
             fontweight="bold", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(HINH / "TS1_duong_danh_doi.png")
plt.show()

## 6. Kiểm định thống kê cho phương án tốt nhất

Nhóm `>15 km` chỉ có 660 chuyến trong tập test, nên một cải thiện 1–2 điểm rất dễ là nhiễu. Phải
có khoảng tin cậy trước khi kết luận.

In [ ]:
# Chon phuong an dat tieu chi, uu tien giam nhieu nhat o nhom hiem
ung_vien = []
for cach in CACH_GAN:
    for w in TRONG_SO:
        if w == 1:
            continue
        cot = f"{cach}_w{w}"
        d_chung = (mape(E[cot], E.y) - mape(E.moc, E.y)) * 100
        s = E[E.kmb.astype(str) == ">15"]
        d_hiem = (mape(s.moc, s.y) - mape(s[cot], s.y)) * 100
        ung_vien.append((cach, w, d_hiem, d_chung))

dat = [u for u in ung_vien if u[2] >= 1.0 and u[3] <= 0.15]
print("Phương án đạt tiêu chí (nhóm hiếm giảm ≥1 điểm, toàn tập xấu đi ≤0,15 điểm):")
if dat:
    for c, w, dh, dc in sorted(dat, key=lambda x: -x[2]):
        print(f"  {c:12} w={w:<3} nhóm >15km −{dh:.2f} điểm · toàn tập +{dc:.3f} điểm")
    CACH_TOT, W_TOT = max(dat, key=lambda x: x[2])[:2]
else:
    print("  KHÔNG có phương án nào đạt cả hai điều kiện.")
    CACH_TOT, W_TOT = max(ung_vien, key=lambda x: x[2])[:2]
    print(f"  Lấy phương án giảm nhiều nhất ở nhóm hiếm để xem chi tiết: "
          f"{CACH_TOT} w={W_TOT}")
print(f"\n→ Phương án đem đi kiểm định: {CACH_TOT} · w={W_TOT}")

In [ ]:
COT_TOT = f"{CACH_TOT}_w{W_TOT}"
BK = bang_theo_nhom(E, "moc", COT_TOT, ten_moc="Mốc", ten_moi=f"w={W_TOT}")
in_bang(BK, ["Mốc", f"w={W_TOT}"])

In [ ]:
# ═════════ HÌNH TS2 — chenh lech kem CI theo tung nhom ═════════
b = BK[BK.Nhóm != "TOÀN TẬP"].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9.5, 0.46*len(b) + 2))
yy = np.arange(len(b))[::-1]
for i, r in b.iterrows():
    y = yy[i]
    co_y = (r["CI thấp"] > 0) or (r["CI cao"] < 0)
    mau = GREEN if (co_y and r["Chênh (điểm)"] > 0) else RED if co_y else MUT
    ax.plot([r["CI thấp"], r["CI cao"]], [y, y], color=mau, lw=2.4, alpha=.85)
    ax.plot(r["Chênh (điểm)"], y, "o", color=mau, ms=8)
    ax.text(r["CI cao"] + .06, y, f"n={r['n']:,}", va="center", fontsize=9, color=MUT)
ax.axvline(0, color=INK, lw=1.2)
ax.set_yticks(yy)
ax.set_yticklabels([f"{r['Chia theo']} · {r['Nhóm']}" for _, r in b.iterrows()])
ax.set_xlabel("Chênh lệch MAPE so với model mốc (điểm) — dương = trọng số giúp ích")
tong = BK[BK.Nhóm == "TOÀN TẬP"].iloc[0]
ax.set_title(f"TS2 — Trọng số {CACH_TOT} w={W_TOT} vs model mốc, khoảng tin cậy 95%\n"
             f"Toàn tập: {tong['Chênh (điểm)']:+.2f} điểm "
             f"[{tong['CI thấp']:+.2f}, {tong['CI cao']:+.2f}]",
             fontweight="bold", fontsize=12.5)
fig.tight_layout()
fig.savefig(HINH / "TS2_trong_so_vs_moc.png")
plt.show()

## 7. Lưu kết quả cho notebook tổng hợp

In [ ]:
DA.to_csv(KQ / "A_danh_doi_trong_so.csv", index=False)
BK.to_csv(KQ / "A_kiem_dinh_phuong_an_tot.csv", index=False)

# luu ca du doan cua phuong an tot de 00_TONG_HOP dung lai
np.save(KQ / "A_pred_tot.npy", E[COT_TOT].values)
json.dump({"cach": CACH_TOT, "w": int(W_TOT), "nhanh": bool(NHANH),
           "n_test": int(len(E)),
           "mape_moc": mape(E.moc, E.y), "mape_moi": mape(E[COT_TOT], E.y)},
          open(KQ / "A_cau_hinh.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("Đã lưu vào", KQ.resolve())

if NHANH:
    print("\n" + "!"*72)
    print("!! NHANH = True — số ở trên chỉ để thử, KHÔNG đưa vào báo cáo.")
    print("!! Đặt NHANH = False rồi chạy lại trước khi lấy số.")
    print("!"*72)

## 8. Kết luận cần điền sau khi chạy

Ba câu, trả lời bằng số trong bảng ở mục 5 và 6:

1. Trọng số **có** giảm được sai số ở nhóm chuyến dài và giá cao không, và khoảng tin cậy có loại
   trừ được 0 không?
2. Cái giá phải trả trên toàn tập là bao nhiêu điểm?
3. Hai cách gán chuyến hiếm — theo quãng đường hay theo giá — cách nào cho đánh đổi tốt hơn? Đây là
   câu cần hỏi mentor kèm số cụ thể.

> ⚠️ Nếu không phương án nào đạt tiêu chí, đó **cũng là kết quả** và cần báo cáo đúng như vậy. Kết
> quả âm tính tiết kiệm thời gian cho tuần sau.